# D01 · CRCs and baud rate error

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/02_drivers_protocols/D01_crcs_and_baud_rate_error.ipynb)

**Track:** Drivers and protocols · **Skill axis (sheet):** Drivers and protocols · **Time:** about 50 min

**Skill Hub concepts:** Cyclic redundancy check; Baud rate error

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("D01")

## Map: a CRC is polynomial division over bits

The message is a polynomial with bit coefficients; the CRC is the remainder after dividing by the generator
polynomial, using XOR instead of subtraction. Both ends must agree on **all five** parameters:

| Name | Width | Poly | Init | Reflect in/out | XOR out | Check value of `"123456789"` |
|---|---|---|---|---|---|---|
| CRC-8/SMBUS | 8 | 0x07 | 0x00 | no / no | 0x00 | 0xF4 |
| CRC-16/CCITT-FALSE | 16 | 0x1021 | 0xFFFF | no / no | 0x0000 | 0x29B1 |
| CRC-16/MODBUS | 16 | 0x8005 (reflected 0xA001) | 0xFFFF | yes / yes | 0x0000 | 0x4B37 |
| CRC-32 (Ethernet, zlib) | 32 | 0x04C11DB7 (reflected 0xEDB88320) | 0xFFFFFFFF | yes / yes | 0xFFFFFFFF | 0xCBF43926 |

```
non-reflected (MSB first):  crc ^= byte << (W-8); repeat 8: crc = (crc << 1) ^ poly if top bit else crc << 1
reflected (LSB first):      crc ^= byte;          repeat 8: crc = (crc >> 1) ^ rpoly if bit 0 else crc >> 1
```

UART baud rate (STM32, 16x oversampling): $\mathrm{BRR} = \mathrm{round}(f_{clk}/\text{baud})$,
$\text{baud}_{actual} = f_{clk}/\mathrm{BRR}$, $\text{error} = (\text{baud}_{actual} - \text{baud})/\text{baud}$.
The receiver resynchronizes on each start bit and samples mid-bit, so the error accumulates over one 10-bit frame:
keep each side within about $\pm 2\%$.

## Task 1: two CRC-16 variants in Python

In [ ]:
def crc16_ccitt_false(data: bytes) -> int:
    raise NotImplementedError  # TODO

def crc16_modbus(data: bytes) -> int:
    raise NotImplementedError  # TODO

lab.check("crc16", crc16_ccitt_false, crc16_modbus)

In [ ]:
%%writefile d01_crc16.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/d01.h
// Grade it with lab.check("crc16", "d01_crc16.c"). For C++, name the file .cpp in both places.
#include "d01.h"

uint16_t crc16_ccitt_false(const uint8_t *data, size_t len) {
    (void)data; (void)len;
    return 0; /* TODO: init 0xFFFF, poly 0x1021, MSB first */
}

uint16_t crc16_modbus(const uint8_t *data, size_t len) {
    (void)data; (void)len;
    return 0; /* TODO: init 0xFFFF, reflected poly 0xA001, LSB first */
}

In [ ]:
# lab.check("crc16", "d01_crc16.c")  # uncomment after writing the C/C++ cell above

## Task 2: CRC-32 in C, streaming
`crc32_update` must work across chunks: `crc32_final(crc32_update(crc32_update(INIT, a), b))` equals the CRC of
`a + b`. Use a 256-entry table built at first call (how firmware usually does it), or the bitwise loop.

In [ ]:
%%writefile crc32.c
#include <stdint.h>
#include <stddef.h>
#define CRC32_INIT 0xFFFFFFFFu

uint32_t crc32_update(uint32_t crc, const uint8_t *p, size_t n) {
    (void)p; (void)n; return crc; /* TODO */
}
uint32_t crc32_final(uint32_t crc) { return crc; /* TODO */ }

In [ ]:
lab.check("crc32")

## Task 3: UART divisor and error

In [ ]:
def uart_brr(f_clk, baud):
    """Return (brr, actual_baud, error_percent) for 16x oversampling."""
    raise NotImplementedError  # TODO

lab.check("baud", uart_brr)

In [ ]:
%%writefile d01_baud.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/d01.h
// Grade it with lab.check("baud", "d01_baud.c"). For C++, name the file .cpp in both places.
#include "d01.h"

uint32_t uart_brr(double f_clk, double baud, double *actual, double *error_percent) {
    (void)f_clk; (void)baud;
    *actual = 0; *error_percent = 0;
    return 0; /* TODO */
}

In [ ]:
# lab.check("baud", "d01_baud.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** 16 MHz clock, 115200 baud, 16x oversampling. Baud error in percent (2 decimals, with sign)?

**M2 (Apply).** CRC-8/SMBUS (poly 0x07, init 0) of the single byte `0x01`?

**M3 (Analyze).** Probability that a random corruption of a long frame slips past a good 16-bit CRC (as a fraction)?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** What does a CRC catch that an 8-bit additive checksum misses?
- a) nothing; they are equivalent
- b) reordered bytes and every burst error shorter than the CRC width
- c) errors in the CRC field itself only

**K2 (Evaluate).** Your Python tool and the STM32 disagree on every CRC-16. List what to compare, in order, and how
the "123456789" check value settles it.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `d01.h`).
3. In `roboprep/checks/d01.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()